# Modelo de Machine Learning para Assinatura Premium

Objetivo: prever a categoria de resposta de um usuário a uma oferta de plano premium de streaming.

Este notebook segue uma abordagem profissional para classificação multiclasse:

- validação dos dados e mapeamento da coluna `y` para categorias de negócio;
- divisão treino/teste estratificada;
- pipelines com imputação, remoção de variáveis constantes e padronização;
- comparação de modelos com validação cruzada repetida;
- otimização de hiperparâmetros com `RandomizedSearchCV`;
- avaliação final em holdout com `macro F1`, `balanced accuracy`, relatório de classificação e matriz de confusão;
- análise de importância por permutação;
- salvamento do modelo final e geração de predições para `novos_dados.csv`.

## 1. Dependências

Execute a célula abaixo se o kernel ainda não tiver as bibliotecas necessárias. Em alguns ambientes Linux, a instalação global via `pip` pode ser bloqueada por PEP 668; nesse caso, a própria célula mostra os comandos para criar um ambiente virtual.

In [ ]:
import importlib.util
import subprocess
import sys

required_packages = {
    'pandas': 'pandas',
    'numpy': 'numpy',
    'sklearn': 'scikit-learn',
    'matplotlib': 'matplotlib',
    'seaborn': 'seaborn',
    'joblib': 'joblib',
}

missing = [pkg for module, pkg in required_packages.items() if importlib.util.find_spec(module) is None]

if missing:
    print('Dependências ausentes:', ', '.join(missing))
    print('Tentando instalar no kernel atual...')
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])
    except subprocess.CalledProcessError as exc:
        print('
A instalação automática falhou. Uma alternativa segura é criar um ambiente virtual:')
        print('python3 -m venv .venv')
        print('source .venv/bin/activate')
        print('python -m pip install -U pip pandas numpy scikit-learn matplotlib seaborn joblib ipykernel')
        print('python -m ipykernel install --user --name assinatura-premium --display-name "Python (assinatura-premium)"')
        raise exc
else:
    print('Dependências disponíveis.')

## 2. Imports e Configuração

In [ ]:
from pathlib import Path
from math import prod
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from joblib import dump

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.feature_selection import VarianceThreshold
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import (
    RandomizedSearchCV,
    RepeatedStratifiedKFold,
    StratifiedKFold,
    cross_validate,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

try:
    from IPython.display import display
except ImportError:
    display = print

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
N_JOBS = -1

pd.set_option('display.max_columns', 80)
pd.set_option('display.max_rows', 80)
sns.set_theme(style='whitegrid', context='notebook')

## 3. Caminhos e Mapeamento do Alvo

A coluna `y` possui códigos de 1 a 5. O dicionário abaixo transforma esses códigos nas categorias solicitadas.

In [ ]:
DATA_PATH = Path('empresa_assinatura.csv')
NOVOS_DADOS_PATH = Path('novos_dados.csv')
MODEL_PATH = Path('modelo_assinatura_premium.joblib')
PREDICTIONS_PATH = Path('predicoes_assinatura_premium.csv')

TARGET_COL = 'y'
TARGET_LABELS = {
    1: 'o usuário assinou o plano premium',
    2: 'o usuário não respondeu a oferta via e-mail',
    3: 'o usuário não respondeu a oferta via notificação no app',
    4: 'o usuário mostrou interesse, mas ainda não assinou',
    5: 'o usuário ainda não assinou',
}
TARGET_ORDER = list(TARGET_LABELS.values())

TARGET_LABELS

## 4. Carregamento e Validação Inicial

In [ ]:
df_raw = pd.read_csv(DATA_PATH)
print(f'Dataset carregado: {df_raw.shape[0]} linhas x {df_raw.shape[1]} colunas')
df_raw.head()

In [ ]:
if TARGET_COL not in df_raw.columns:
    raise ValueError(f'A coluna alvo {TARGET_COL!r} não foi encontrada no CSV.')

unknown_codes = sorted(set(df_raw[TARGET_COL].dropna().unique()) - set(TARGET_LABELS))
if unknown_codes:
    raise ValueError(f'Códigos não mapeados na coluna y: {unknown_codes}')

df = df_raw.copy()
df['y_categoria'] = df[TARGET_COL].map(TARGET_LABELS)

feature_cols = [col for col in df.columns if col not in [TARGET_COL, 'y_categoria']]
X = df[feature_cols].copy()
y = df['y_categoria'].copy()

quality_summary = pd.DataFrame({
    'linhas': [df.shape[0]],
    'variaveis_preditoras': [len(feature_cols)],
    'nulos_total': [int(df.isna().sum().sum())],
    'linhas_duplicadas': [int(df.duplicated().sum())],
    'classes_target': [y.nunique()],
})

display(quality_summary)
display(df[[TARGET_COL, 'y_categoria']].drop_duplicates().sort_values(TARGET_COL))

## 5. Análise Exploratória

In [ ]:
target_summary = (
    df['y_categoria']
    .value_counts()
    .reindex(TARGET_ORDER)
    .rename_axis('categoria')
    .reset_index(name='quantidade')
)
target_summary['percentual'] = (target_summary['quantidade'] / len(df) * 100).round(2)
display(target_summary)

fig, ax = plt.subplots(figsize=(12, 4))
sns.barplot(data=target_summary, x='categoria', y='quantidade', ax=ax, color='#357ABD')
ax.set_title('Distribuição das categorias do alvo')
ax.set_xlabel('')
ax.set_ylabel('Quantidade')
ax.tick_params(axis='x', rotation=35)
plt.tight_layout()
plt.show()

In [ ]:
feature_stats = X.agg(['mean', 'std', 'min', 'max']).T

display(feature_stats.describe().round(2))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(feature_stats['mean'], bins=30, ax=axes[0], color='#2A9D8F')
axes[0].set_title('Distribuição das médias das features')
axes[0].set_xlabel('Média')

sns.histplot(feature_stats['std'], bins=30, ax=axes[1], color='#E76F51')
axes[1].set_title('Distribuição dos desvios-padrão das features')
axes[1].set_xlabel('Desvio-padrão')

plt.tight_layout()
plt.show()

## 6. Separação Treino/Teste

O conjunto de teste fica reservado para a avaliação final. A validação cruzada e o tuning acontecem apenas no treino, reduzindo vazamento de informação.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

split_summary = pd.DataFrame({
    'amostra': ['treino', 'teste'],
    'linhas': [len(X_train), len(X_test)],
    'classes': [y_train.nunique(), y_test.nunique()],
})
display(split_summary)

display(
    pd.concat(
        [
            y_train.value_counts(normalize=True).reindex(TARGET_ORDER).rename('treino'),
            y_test.value_counts(normalize=True).reindex(TARGET_ORDER).rename('teste'),
        ],
        axis=1,
    ).mul(100).round(2)
)

## 7. Pipeline de Pré-processamento e Modelos Candidatos

In [ ]:
numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
non_numeric_features = sorted(set(X_train.columns) - set(numeric_features))

if non_numeric_features:
    raise TypeError(f'Este notebook espera variáveis numéricas. Colunas não numéricas: {non_numeric_features}')

numeric_preprocess = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('variance_threshold', VarianceThreshold(threshold=0.0)),
        ('scaler', StandardScaler()),
    ]
)

preprocess = ColumnTransformer(
    transformers=[('numeric', numeric_preprocess, numeric_features)],
    remainder='drop',
)


def build_pipeline(model):
    return Pipeline(
        steps=[
            ('preprocess', preprocess),
            ('model', model),
        ]
    )

models = {
    'Dummy estratificado': DummyClassifier(strategy='stratified', random_state=RANDOM_STATE),
    'Regressão Logística': LogisticRegression(max_iter=5000, class_weight='balanced', random_state=RANDOM_STATE),
    'Random Forest': RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=N_JOBS),
    'Extra Trees': ExtraTreesClassifier(n_estimators=400, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=N_JOBS),
    'HistGradientBoosting': HistGradientBoostingClassifier(random_state=RANDOM_STATE, early_stopping=True),
}

list(models)

## 8. Comparação com Validação Cruzada

A métrica principal será `macro F1`, porque ela dá o mesmo peso para todas as classes. Também acompanhamos acurácia e `balanced accuracy`.

In [ ]:
scoring = {
    'accuracy': 'accuracy',
    'balanced_accuracy': 'balanced_accuracy',
    'f1_macro': 'f1_macro',
}

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=2, random_state=RANDOM_STATE)
rows = []

for name, model in models.items():
    print(f'Avaliando: {name}')
    pipe = build_pipeline(model)
    scores = cross_validate(
        pipe,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        return_train_score=True,
        n_jobs=1,
    )

    row = {
        'modelo': name,
        'fit_time_mean': scores['fit_time'].mean(),
    }
    for metric_name in scoring:
        row[f'train_{metric_name}_mean'] = scores[f'train_{metric_name}'].mean()
        row[f'test_{metric_name}_mean'] = scores[f'test_{metric_name}'].mean()
        row[f'test_{metric_name}_std'] = scores[f'test_{metric_name}'].std()
    rows.append(row)

cv_results = pd.DataFrame(rows).sort_values('test_f1_macro_mean', ascending=False)
display(cv_results.round(4))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
plot_df = cv_results.sort_values('test_f1_macro_mean', ascending=True)
ax.barh(plot_df['modelo'], plot_df['test_f1_macro_mean'], xerr=plot_df['test_f1_macro_std'], color='#4C78A8')
ax.set_title('Comparação de modelos por validação cruzada')
ax.set_xlabel('Macro F1 médio no treino validado')
ax.set_xlim(0, max(0.05, plot_df['test_f1_macro_mean'].max() + 0.05))
plt.tight_layout()
plt.show()

## 9. Otimização de Hiperparâmetros

O tuning é aplicado ao melhor modelo real da comparação anterior, ignorando o baseline `Dummy`.

In [ ]:
search_spaces = {
    'Regressão Logística': {
        'model__C': np.logspace(-3, 2, 12).tolist(),
        'model__class_weight': [None, 'balanced'],
    },
    'Random Forest': {
        'model__n_estimators': [250, 400, 600],
        'model__max_depth': [None, 6, 10, 16, 24],
        'model__min_samples_split': [2, 5, 10],
        'model__min_samples_leaf': [1, 2, 4],
        'model__max_features': ['sqrt', 'log2', 0.5],
        'model__class_weight': [None, 'balanced'],
    },
    'Extra Trees': {
        'model__n_estimators': [300, 500, 800],
        'model__max_depth': [None, 6, 10, 16, 24],
        'model__min_samples_split': [2, 5, 10],
        'model__min_samples_leaf': [1, 2, 4],
        'model__max_features': ['sqrt', 'log2', 0.5],
        'model__class_weight': [None, 'balanced'],
    },
    'HistGradientBoosting': {
        'model__max_iter': [100, 200, 300],
        'model__learning_rate': [0.03, 0.06, 0.10, 0.15],
        'model__max_leaf_nodes': [15, 31, 63],
        'model__min_samples_leaf': [10, 20, 40],
        'model__l2_regularization': [0.0, 0.01, 0.1, 1.0],
    },
}

real_model_results = cv_results[cv_results['modelo'] != 'Dummy estratificado'].copy()
best_model_name = real_model_results.iloc[0]['modelo']
print(f'Modelo selecionado para tuning: {best_model_name}')


def param_space_size(param_space):
    return int(prod(len(values) for values in param_space.values()))

param_space = search_spaces[best_model_name]
n_iter = min(30, param_space_size(param_space))

search_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
search = RandomizedSearchCV(
    estimator=build_pipeline(clone(models[best_model_name])),
    param_distributions=param_space,
    n_iter=n_iter,
    scoring='f1_macro',
    cv=search_cv,
    random_state=RANDOM_STATE,
    n_jobs=1,
    refit=True,
    return_train_score=True,
    verbose=1,
)

search.fit(X_train, y_train)

print(f'Melhor macro F1 médio em CV: {search.best_score_:.4f}')
print('Melhores hiperparâmetros:')
print(search.best_params_)

In [ ]:
search_results = pd.DataFrame(search.cv_results_).sort_values('rank_test_score')
cols_to_show = [
    'rank_test_score',
    'mean_test_score',
    'std_test_score',
    'mean_train_score',
    'params',
]
display(search_results[cols_to_show].head(10))

## 10. Avaliação Final no Conjunto de Teste

In [ ]:
best_estimator = search.best_estimator_
y_pred = best_estimator.predict(X_test)

test_metrics = pd.DataFrame([
    {
        'accuracy': accuracy_score(y_test, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_test, y_pred),
        'f1_macro': f1_score(y_test, y_pred, average='macro'),
    }
])

display(test_metrics.round(4))
print(classification_report(y_test, y_pred, labels=TARGET_ORDER, zero_division=0))

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=TARGET_ORDER)
fig, ax = plt.subplots(figsize=(11, 8))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=TARGET_ORDER)
disp.plot(ax=ax, cmap='Blues', colorbar=False, xticks_rotation=35)
ax.set_title('Matriz de confusão - conjunto de teste')
plt.tight_layout()
plt.show()

In [ ]:
error_df = pd.DataFrame({'real': y_test, 'previsto': y_pred})
error_df = error_df[error_df['real'] != error_df['previsto']]

if error_df.empty:
    print('Nenhum erro no conjunto de teste.')
else:
    most_common_errors = (
        error_df
        .groupby(['real', 'previsto'])
        .size()
        .sort_values(ascending=False)
        .reset_index(name='quantidade')
        .head(10)
    )
    display(most_common_errors)

## 11. Confiança das Predições

Modelos probabilísticos permitem identificar casos incertos. Isso ajuda a priorizar revisão manual, novas campanhas ou coleta de mais dados.

In [ ]:
if hasattr(best_estimator, 'predict_proba'):
    proba = best_estimator.predict_proba(X_test)
    confidence_df = pd.DataFrame({
        'real': y_test.to_numpy(),
        'previsto': y_pred,
        'confianca': proba.max(axis=1),
        'acertou': y_test.to_numpy() == y_pred,
    })

    display(confidence_df.groupby('acertou')['confianca'].describe().round(3))

    fig, ax = plt.subplots(figsize=(9, 4))
    sns.histplot(data=confidence_df, x='confianca', hue='acertou', bins=20, multiple='layer', ax=ax)
    ax.set_title('Distribuição da confiança das predições')
    ax.set_xlabel('Maior probabilidade prevista')
    plt.tight_layout()
    plt.show()

    display(confidence_df.sort_values('confianca').head(10))
else:
    print('O modelo escolhido não expõe predict_proba().')

## 12. Importância das Variáveis

Como as features estão anonimizadas, a importância por permutação mostra quais colunas mais afetam a métrica do modelo no conjunto de teste.

In [ ]:
perm = permutation_importance(
    best_estimator,
    X_test,
    y_test,
    scoring='f1_macro',
    n_repeats=8,
    random_state=RANDOM_STATE,
    n_jobs=N_JOBS,
)

importance_df = (
    pd.DataFrame({
        'feature': feature_cols,
        'importance_mean': perm.importances_mean,
        'importance_std': perm.importances_std,
    })
    .sort_values('importance_mean', ascending=False)
    .reset_index(drop=True)
)

display(importance_df.head(20).round(4))

fig, ax = plt.subplots(figsize=(9, 7))
top_importance = importance_df.head(20).sort_values('importance_mean', ascending=True)
ax.barh(top_importance['feature'], top_importance['importance_mean'], xerr=top_importance['importance_std'], color='#2A9D8F')
ax.set_title('Top 20 variáveis por importância de permutação')
ax.set_xlabel('Queda média no macro F1')
plt.tight_layout()
plt.show()

## 13. Treino Final e Salvamento do Modelo

Depois da avaliação, treinamos uma versão final com todos os dados disponíveis e salvamos o pipeline completo. O objeto salvo inclui o modelo, as colunas usadas e o mapeamento das classes.

In [ ]:
final_model = clone(best_estimator)
final_model.fit(X, y)

model_package = {
    'model': final_model,
    'feature_cols': feature_cols,
    'target_col': TARGET_COL,
    'target_labels': TARGET_LABELS,
    'target_order': TARGET_ORDER,
    'selected_model_name': best_model_name,
    'best_params': search.best_params_,
    'holdout_metrics': test_metrics.iloc[0].to_dict(),
}

dump(model_package, MODEL_PATH)
print(f'Modelo salvo em: {MODEL_PATH.resolve()}')

## 14. Predição em Novos Dados

Se `novos_dados.csv` existir, a célula abaixo gera as categorias previstas e salva o resultado em `predicoes_assinatura_premium.csv`.

In [ ]:
if NOVOS_DADOS_PATH.exists():
    novos_df = pd.read_csv(NOVOS_DADOS_PATH)
    novos_features = novos_df.drop(columns=[TARGET_COL], errors='ignore').copy()

    missing_cols = sorted(set(feature_cols) - set(novos_features.columns))
    extra_cols = sorted(set(novos_features.columns) - set(feature_cols))

    if missing_cols:
        raise ValueError(f'Novos dados não possuem colunas esperadas: {missing_cols[:10]}')

    if extra_cols:
        print(f'Colunas extras ignoradas: {extra_cols[:10]}')

    novos_features = novos_features[feature_cols]
    pred_new = final_model.predict(novos_features)

    prediction_df = novos_df.copy()
    prediction_df['predicao_y_categoria'] = pred_new

    if hasattr(final_model, 'predict_proba'):
        proba_new = final_model.predict_proba(novos_features)
        label_to_code = {label: code for code, label in TARGET_LABELS.items()}
        proba_cols = [f'prob_y_{label_to_code[label]}' for label in final_model.classes_]
        prediction_df = pd.concat(
            [prediction_df, pd.DataFrame(proba_new, columns=proba_cols, index=prediction_df.index)],
            axis=1,
        )

    prediction_df.to_csv(PREDICTIONS_PATH, index=False)
    display(prediction_df.head())
    print(f'Predições salvas em: {PREDICTIONS_PATH.resolve()}')
else:
    print(f'Arquivo {NOVOS_DADOS_PATH} não encontrado. Pule esta etapa ou informe novos dados.')

## 15. Função Reutilizável de Predição

In [ ]:
def prever_assinatura(dados, modelo=final_model):
    """Recebe um DataFrame ou caminho de CSV e retorna a predição de categoria para cada linha."""
    if isinstance(dados, (str, Path)):
        dados = pd.read_csv(dados)

    dados_features = dados.drop(columns=[TARGET_COL], errors='ignore').copy()
    missing_cols = sorted(set(feature_cols) - set(dados_features.columns))
    if missing_cols:
        raise ValueError(f'Dados sem colunas esperadas: {missing_cols[:10]}')

    dados_features = dados_features[feature_cols]
    pred = modelo.predict(dados_features)

    resultado = dados.copy()
    resultado['predicao_y_categoria'] = pred

    if hasattr(modelo, 'predict_proba'):
        proba_pred = modelo.predict_proba(dados_features)
        label_to_code = {label: code for code, label in TARGET_LABELS.items()}
        proba_cols = [f'prob_y_{label_to_code[label]}' for label in modelo.classes_]
        resultado = pd.concat(
            [resultado, pd.DataFrame(proba_pred, columns=proba_cols, index=resultado.index)],
            axis=1,
        )

    return resultado

# Exemplo:
# prever_assinatura('novos_dados.csv')

## Observações Profissionais

- Se o objetivo de negócio virar apenas "assinou" vs. "não assinou", transforme o alvo em binário e use métricas como `ROC AUC`, `PR AUC`, recall da classe positiva e análise de limiar.
- Como as variáveis estão anonimizadas, a etapa de engenharia de atributos fica limitada. Com nomes reais, vale criar features de comportamento, engajamento, histórico de consumo, resposta a campanhas e recência.
- Em produção, monitore drift das variáveis, queda de confiança, distribuição das classes previstas e performance por segmento de usuário.